# Modeling HUPA0003's as a multivariate time series using an RNN, glucose, rapid acting insulin are considered as input at each step t  

In [93]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

## Get data 

In [94]:
data = pd.read_csv('../Data/Preprocessed/HUPA0003P.csv', sep=';')
data['time'] = pd.to_datetime(data['time'])

In [95]:
data

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2018-06-13 21:40:00,137.666667,7.32088,70.692308,8.0,0.075,0.0,0.0
1,2018-06-13 21:45:00,137.000000,11.19664,90.333333,17.0,0.075,0.0,0.0
2,2018-06-13 21:50:00,136.333333,8.18216,84.677419,0.0,0.075,0.0,0.0
3,2018-06-13 21:55:00,135.666667,8.82812,89.727273,8.0,0.075,0.0,0.0
4,2018-06-13 22:00:00,135.000000,6.67492,91.235294,0.0,0.075,0.0,0.0
...,...,...,...,...,...,...,...,...
3765,2018-06-26 23:25:00,160.666667,5.59674,68.875000,0.0,0.075,0.0,0.0
3766,2018-06-26 23:30:00,158.000000,5.48700,71.621622,0.0,0.075,0.0,0.0
3767,2018-06-26 23:35:00,159.000000,5.70648,71.692308,0.0,0.075,0.0,0.0
3768,2018-06-26 23:40:00,160.000000,5.48700,70.346154,0.0,0.075,0.0,0.0


## Architecture 

- input_size = 1 (Size of vector at each time step)
- hidden_size = 4 (Number of recurrent neurons)
- batch_size = 12

In [96]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

Prepare data 

Import our own class 

In [97]:
import sys
sys.path.append("..")
from scripts.window_regressor import Window_Regressor

In [98]:
batch_size = 12
input_size = 1
hidden_size = 6
window_size = 4

In [99]:
train = Window_Regressor(data['glucose'][:-batch_size], window_size=window_size, horizon=1).generate_data_set()
test = Window_Regressor(data['glucose'][-batch_size:], window_size=window_size, horizon=1).generate_data_set()

In [100]:
train_data = torch.tensor(train.values, dtype=torch.float32)
test_data = torch.tensor(test.values, dtype=torch.float32)

In [101]:
train_dataset = TensorDataset(train_data)
test_dataset = TensorDataset(test_data)
train_loader = DataLoader(train_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

Define the model

In [102]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.linear_layer = nn.Linear(in_features=hidden_size, out_features=1)
        
    
    def forward(self, x):
        output_recurrent_layer, _ = self.rnn(x)
        prediction = self.linear_layer(output_recurrent_layer[:, -1, :]) 
        return prediction.squeeze(-1)
        

In [103]:
model = RNN(input_size=input_size, hidden_size=hidden_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.MSELoss()
epochs = 300

Train the model 

In [ ]:
train_losses = []
for e in range(epochs):
    epoch_losses = []
    for x in train_loader:
        batch = x[0]                #  4 inputs + 1 target
        x_in = batch[:, :-1]        # (batch_size, 4)
        y = batch[:, -1]            # (batch_size,)
        x_in = x_in.unsqueeze(-1)
        y_pred = model(x_in)        # (batch_size,)
        loss = loss_function(y_pred, y)
        epoch_losses.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    train_losses.append(avg_loss)
    if e % 10 == 0:
        print(f'Epoch {e}/{epochs}, Average Loss: {avg_loss:.4f}')

Epoch 0/300, Average Loss: 23911.6081
Epoch 10/300, Average Loss: 18746.8915
Epoch 20/300, Average Loss: 14544.9988
Epoch 30/300, Average Loss: 11184.3031
Epoch 40/300, Average Loss: 8608.8768
Epoch 50/300, Average Loss: 6655.6147
Epoch 60/300, Average Loss: 5170.9806
Epoch 70/300, Average Loss: 4054.9844
Epoch 80/300, Average Loss: 3212.7969
Epoch 90/300, Average Loss: 2565.8523
Epoch 100/300, Average Loss: 2062.8727
Epoch 110/300, Average Loss: 1670.7471
Epoch 120/300, Average Loss: 1365.7188
Epoch 130/300, Average Loss: 1129.9336
Epoch 140/300, Average Loss: 949.1544
Epoch 150/300, Average Loss: 806.3412
Epoch 160/300, Average Loss: 695.5889
Epoch 170/300, Average Loss: 608.5875
Epoch 180/300, Average Loss: 539.7410
Epoch 190/300, Average Loss: 485.2135
Epoch 200/300, Average Loss: 441.4828
Epoch 210/300, Average Loss: 403.6052
Epoch 220/300, Average Loss: 372.2125
Epoch 230/300, Average Loss: 346.0001
Epoch 240/300, Average Loss: 322.5519
Epoch 250/300, Average Loss: 301.7317
Epoch

In [ ]:
def predict_recursive(model, last_window, n_steps):
    model.eval()
    window = last_window.clone().detach()  # (L,)
    predictions = []

    for _ in range(n_steps):
        # Preparar entrada: (1, L, 1) → batch=1, seq=L, features=1
        x = window.unsqueeze(0).unsqueeze(-1)  # (1, L, 1)
        
        with torch.no_grad():
            pred = model(x)  # (1,)
        
        pred_val = pred.item()
        predictions.append(pred_val)
        
        # Actualizar ventana: quitar el primero, añadir la predicción
        window = torch.cat([window[1:], torch.tensor([pred_val])])

    return predictions

In [107]:
with torch.no_grad():
    for x in test_loader:
        batch = x[0]
        x = batch[:, :-1]
        y = batch[:, -1]
        output_model = model(x.unsqueeze(-1))
        print(batch)
        print(output_model)
        print(y)

tensor([[172.6667, 173.3333, 174.0000, 171.3333, 168.6667],
        [173.3333, 174.0000, 171.3333, 168.6667, 166.0000],
        [174.0000, 171.3333, 168.6667, 166.0000, 163.3333],
        [171.3333, 168.6667, 166.0000, 163.3333, 160.6667],
        [168.6667, 166.0000, 163.3333, 160.6667, 158.0000],
        [166.0000, 163.3333, 160.6667, 158.0000, 159.0000],
        [163.3333, 160.6667, 158.0000, 159.0000, 160.0000],
        [160.6667, 158.0000, 159.0000, 160.0000, 161.0000]])
tensor([168.1029, 166.2826, 164.3614, 162.4016, 160.3848, 158.2940, 160.8184,
        161.3319])
tensor([168.6667, 166.0000, 163.3333, 160.6667, 158.0000, 159.0000, 160.0000,
        161.0000])


In [109]:
test

,Predictor_0,Predictor_1,Predictor_2,Predictor_3,Target
0,172.666667,173.333333,174.000000,171.333333,168.666667
1,173.333333,174.000000,171.333333,168.666667,166.000000
2,174.000000,171.333333,168.666667,166.000000,163.333333
3,171.333333,168.666667,166.000000,163.333333,160.666667
4,168.666667,166.000000,163.333333,160.666667,158.000000
5,166.000000,163.333333,160.666667,158.000000,159.000000
6,163.333333,160.666667,158.000000,159.000000,160.000000
7,160.666667,158.000000,159.000000,160.000000,161.000000


In [111]:
predict_recursive(model, last_window=torch.tensor([172.6667, 173.3333, 174.0000, 171.3333]), n_steps=5)

[168.10292053222656,
 165.69773864746094,
 164.23883056640625,
 163.49452209472656,
 163.2037811279297]